# talk-tag Colab quickstart

This notebook uses the fixed adapter deployment path and annotates a `.cha` file.

Deployment chain used at runtime:
1. `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` (base + tokenizer)
2. `chat_tokens.json` token injection (+ resize when needed)
3. `mash-mash/talkbank-morphosyntax-annotator-final-recon_full_comp_preserve_final_seed3407` adapter
4. Greedy decoding (`do_sample=False`)

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(
    args: list[str],
    check: bool = True,
    env: dict[str, str] | None = None,
) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(args))
    result = subprocess.run(args, text=True, capture_output=True, env=env)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            "Command failed ({}): {}".format(
                result.returncode,
                " ".join(args),
            )
        )
    return result


cwd = Path.cwd()
repo_src = None
for base in [cwd, *cwd.parents]:
    candidate = base / "src" / "talk_tag"
    if candidate.exists():
        repo_src = candidate.parent
        break

talk_tag_cmd = ["talk-tag"]
talk_tag_env = None

if shutil.which("talk-tag"):
    print("Using installed talk-tag executable.")
else:
    install_attempts = [
        [sys.executable, "-m", "pip", "install", "-q", "talk-tag[runtime]"],
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[runtime]"],
    ]
    for cmd in install_attempts:
        install_result = run(cmd, check=False)
        if install_result.returncode == 0 and shutil.which("talk-tag"):
            break

    if shutil.which("talk-tag"):
        print("Installed talk-tag executable.")
    elif repo_src is not None:
        talk_tag_cmd = [sys.executable, "-m", "talk_tag"]
        talk_tag_env = os.environ.copy()
        current_pythonpath = talk_tag_env.get("PYTHONPATH", "")
        talk_tag_env["PYTHONPATH"] = (
            str(repo_src)
            if not current_pythonpath
            else str(repo_src) + os.pathsep + current_pythonpath
        )
        print("Falling back to local source: python -m talk_tag")
    else:
        raise RuntimeError(
            "talk-tag CLI is not available and local src/talk_tag was not found."
        )


def run_talk_tag(*args: str, check: bool = False) -> subprocess.CompletedProcess[str]:
    return run([*talk_tag_cmd, *args], check=check, env=talk_tag_env)

## 1) Environment and HF auth

You need Hugging Face access to both deployment repos. Set `HF_TOKEN` before pull/annotate.

In [ ]:
import os
import platform

print(f"Python: {platform.python_version()}")
print(f"HF_TOKEN set: {bool(os.environ.get('HF_TOKEN'))}")
print("Required repos:")
print("  - unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit")
print("  - mash-mash/talkbank-morphosyntax-annotator-final-recon_full_comp_preserve_final_seed3407")

## 2) Runtime checks and model pull

In [ ]:
doctor_result = run_talk_tag("doctor", "--json")
pull_result = run_talk_tag("model", "pull", "--device", "auto", "--json")

ready_for_annotation = pull_result.returncode == 0
print(f"Ready for annotation: {ready_for_annotation}")
if not ready_for_annotation:
    print("Model pull failed. Run 'talk-tag doctor --json' above to diagnose.")

## 3) Build a minimal `.cha` sample

In [ ]:
from pathlib import Path

work_dir = Path("demo_workspace").resolve()
input_dir = work_dir / "sample_in"
output_dir = work_dir / "sample_out"
input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

sample_cha = """@Begin
@Participants:	CHI Child, INV Investigator
*INV:	where are you going ?
*CHI:	he go school yesterday .
@End
"""
(input_dir / "sample.cha").write_text(sample_cha, encoding="utf-8")
print(f"Working directory: {work_dir}")
print((input_dir / "sample.cha").read_text(encoding="utf-8"))

## 4) Annotate the sample

In [ ]:
if ready_for_annotation:
    annotate_result = run_talk_tag(
        "annotate",
        "--input-dir",
        str(input_dir),
        "--output-dir",
        str(output_dir),
        "--target-speaker",
        "*CHI",
        "--device",
        "auto",
    )
    output_file = output_dir / "sample.cha"
    report_path = output_dir / "_talk_tag_report.json"

    if annotate_result.returncode == 0 and output_file.exists():
        print(output_file.read_text(encoding="utf-8"))
    elif annotate_result.returncode == 0:
        print("Annotation command completed, but no output file was produced.")
    else:
        print("Annotation failed. Check command output above.")

    if report_path.exists():
        print(report_path.read_text(encoding="utf-8"))
else:
    print("Skipped annotation because model pull was not successful.")

## Notes

- Runtime annotation inputs are `.cha` and `.jsonl` only.
- For `.jsonl`, use `--speaker-field` and `--text-field`.